In [ ]:
"""
Model/Mode Similarity Distribution Comparison Notebook
Compares similarity distributions for each model/mode combination against correct SCAR distribution.
Generates separate histograms for target-source and source-to-source similarity.
"""

import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
from sentence_transformers import SentenceTransformer
import os
import sys
import re

# Get the notebook's directory and add parent directory to path for config import
current_dir = os.getcwd()
llm_dir = None

# Strategy 1: Check if we're already in the LLM directory
if os.path.exists(os.path.join(current_dir, 'core', 'config.py')):
    llm_dir = current_dir
# Strategy 2: Check if we're in the analysis subdirectory
elif os.path.basename(current_dir) == 'analysis' and os.path.exists(os.path.join(os.path.dirname(current_dir), 'core', 'config.py')):
    llm_dir = os.path.dirname(current_dir)
# Strategy 3: Walk up the directory tree to find LLM directory
else:
    search_dir = os.path.abspath(current_dir)
    while search_dir:
        if os.path.exists(os.path.join(search_dir, 'core', 'config.py')):
            llm_dir = search_dir
            break
        parent = os.path.dirname(search_dir)
        if parent == search_dir:  # Reached filesystem root
            break
        search_dir = parent

if llm_dir:
    sys.path.insert(0, llm_dir)
    notebook_dir = llm_dir
else:
    notebook_dir = os.path.join(os.path.dirname(current_dir) if os.path.basename(current_dir) == 'analysis' else current_dir)
    if os.path.basename(notebook_dir) != 'LLM':
        potential_llm = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'analysis' else notebook_dir
        if os.path.exists(os.path.join(potential_llm, 'core', 'config.py')):
            llm_dir = potential_llm
            sys.path.insert(0, llm_dir)
            notebook_dir = llm_dir

# Load environment variables
from dotenv import load_dotenv
env_paths = [
    os.path.join(os.path.dirname(notebook_dir), '.env'),
    os.path.join(os.path.dirname(os.path.dirname(notebook_dir)), '.env'),
    os.path.join(notebook_dir, '.env'),
]
for env_path in env_paths:
    if os.path.exists(env_path):
        load_dotenv(env_path)
        break

# Import from core.config
from core.config import RESULTS_DIR, EMBEDDINGS_PATH, EMBEDDING_MODEL, SCAR_PATH, SIMILARITY_THRESHOLD

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("Imports completed successfully!")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
# Load aggregated results and SCAR dataset
print("=" * 70)
print("Loading Data")
print("=" * 70)

# Load aggregated results
targetonly_path = os.path.join(RESULTS_DIR, 'all_results_targetonly.csv')
withsub_path = os.path.join(RESULTS_DIR, 'all_results_withsub.csv')

dfs = []
if os.path.exists(targetonly_path):
    df_to = pd.read_csv(targetonly_path)
    df_to['mode'] = 'targetonly'  # Ensure mode column exists
    dfs.append(df_to)
    print(f"Loaded: {targetonly_path} ({len(df_to)} records)")

if os.path.exists(withsub_path):
    df_ws = pd.read_csv(withsub_path)
    df_ws['mode'] = 'withsub'  # Ensure mode column exists
    dfs.append(df_ws)
    print(f"Loaded: {withsub_path} ({len(df_ws)} records)")

if not dfs:
    raise FileNotFoundError("No aggregated results found! Please run aggregate_results.py first.")

all_results_df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal records: {len(all_results_df)}")
print(f"Models: {all_results_df['model'].nunique()}")
print(f"Modes: {all_results_df['mode'].unique().tolist()}")

# Load SCAR dataset
print(f"\nLoading SCAR dataset from: {SCAR_PATH}")
scar_df = pd.read_csv(SCAR_PATH)
print(f"Loaded {len(scar_df)} target-source pairs")

# Initialize SentenceTransformer embedder
print(f"\nInitializing SentenceTransformer embedder: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("Embedder ready!")

# Create output directory
output_dir = os.path.join(RESULTS_DIR, 'similarity_analysis', 'model_comparisons')
os.makedirs(output_dir, exist_ok=True)
print(f"\nOutput directory: {output_dir}")

## Step 1: Compute Correct SCAR Distributions

Compute the correct SCAR target-source (average method) and source-to-source similarity distributions to use as baselines for comparison.

In [ ]:
# Compute correct SCAR target-source similarity (average method)
print("=" * 70)
print("Computing Correct SCAR Target-Source Similarity (Average Method)")
print("=" * 70)

# Group by target
target_groups = scar_df.groupby('system_a')['system_b'].apply(list).reset_index()
target_groups.columns = ['target', 'sources']

target_avg_similarities = []

for idx, row in target_groups.iterrows():
    target = str(row['target']).strip()
    sources = [str(s).strip() for s in row['sources'] if str(s).strip() and str(s) != 'nan']
    
    if not target or target == 'nan' or len(sources) == 0:
        continue
    
    # Compute similarity for each source of this target
    target_similarities = []
    for source in sources:
        try:
            target_emb = embedder.encode([target.lower()])[0]
            source_emb = embedder.encode([source.lower()])[0]
            
            similarity = np.dot(target_emb, source_emb) / (
                np.linalg.norm(target_emb) * np.linalg.norm(source_emb) + 1e-8
            )
            target_similarities.append(float(similarity))
        except Exception as e:
            print(f"Error processing {target} vs {source}: {e}")
            continue
    
    if len(target_similarities) > 0:
        # Average similarities for this target
        avg_sim = np.mean(target_similarities)
        target_avg_similarities.append({
            'target': target,
            'avg_similarity': avg_sim,
            'num_sources': len(target_similarities)
        })
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(target_groups)} targets...")

scar_sim_df_recalculated = pd.DataFrame(target_avg_similarities)
print(f"\nComputed average similarities for {len(scar_sim_df_recalculated)} targets")
print(f"Mean: {scar_sim_df_recalculated['avg_similarity'].mean():.4f}")
print(f"Median: {scar_sim_df_recalculated['avg_similarity'].median():.4f}")

In [ ]:
# Compute correct SCAR source-to-source similarity
print("=" * 70)
print("Computing Correct SCAR Source-to-Source Similarity")
print("=" * 70)

# Find targets with multiple sources
multi_source_targets = target_groups[target_groups['sources'].apply(lambda x: len([s for s in x if str(s).strip() and str(s) != 'nan']) >= 2)]
print(f"\nFound {len(multi_source_targets)} targets with multiple sources")

# Compute source-to-source similarity
print(f"\nComputing source-to-source similarity using {EMBEDDING_MODEL}...")
source_to_source_similarities = []

for idx, row in multi_source_targets.iterrows():
    target = row['target']
    sources = [str(s).strip() for s in row['sources'] if str(s).strip() and str(s) != 'nan']
    
    if len(sources) < 2:
        continue
    
    # Compute pairwise similarity between all sources for this target
    for i, source1 in enumerate(sources):
        for j, source2 in enumerate(sources):
            if i < j:  # Only compute each pair once
                try:
                    source1_emb = embedder.encode([source1.lower()])[0]
                    source2_emb = embedder.encode([source2.lower()])[0]
                    
                    similarity = np.dot(source1_emb, source2_emb) / (
                        np.linalg.norm(source1_emb) * np.linalg.norm(source2_emb) + 1e-8
                    )
                    
                    source_to_source_similarities.append({
                        'target': target,
                        'source1': source1,
                        'source2': source2,
                        'similarity': float(similarity),
                        'num_sources_for_target': len(sources)
                    })
                except Exception as e:
                    print(f"Error processing {target}: {e}")
                    continue
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(multi_source_targets)} targets...")

source_sim_df = pd.DataFrame(source_to_source_similarities)
print(f"\nComputed {len(source_sim_df)} source-to-source similarity pairs")
print(f"Mean: {source_sim_df['similarity'].mean():.4f}")
print(f"Median: {source_sim_df['similarity'].median():.4f}")

## Step 2: Extract Model/Mode Combinations

Get all unique combinations of model and mode from the aggregated results.

In [ ]:
# Extract unique model/mode combinations
print("=" * 70)
print("Extracting Model/Mode Combinations")
print("=" * 70)

model_mode_combinations = all_results_df[['model', 'mode']].drop_duplicates().sort_values(['model', 'mode'])
print(f"\nFound {len(model_mode_combinations)} unique model/mode combinations:")
for idx, row in model_mode_combinations.iterrows():
    print(f"  {row['model']} - {row['mode']}")

# Helper function to sanitize model names for filenames
def sanitize_filename(name):
    """Replace special characters in model names for safe filenames"""
    return re.sub(r'[<>:"/\\|?*]', '_', name)

print(f"\nTotal combinations to process: {len(model_mode_combinations)}")

## Step 3: Compute Model/Mode Similarity Distributions and Create Visualizations

For each model/mode combination, compute target-source and source-to-source similarity distributions, then create comparison histograms.

In [ ]:
# Process each model/mode combination
print("=" * 70)
print("Processing Model/Mode Combinations")
print("=" * 70)

for combo_idx, (_, combo_row) in enumerate(model_mode_combinations.iterrows(), 1):
    model_name = combo_row['model']
    mode_name = combo_row['mode']
    
    print(f"\n{'='*70}")
    print(f"Processing {combo_idx}/{len(model_mode_combinations)}: {model_name} - {mode_name}")
    print(f"{'='*70}")
    
    # Filter results for this model/mode
    model_results = all_results_df[
        (all_results_df['model'] == model_name) & 
        (all_results_df['mode'] == mode_name)
    ].copy()
    
    print(f"Found {len(model_results)} records for {model_name} - {mode_name}")
    
    # ========================================================================
    # TARGET-SOURCE SIMILARITY (Average Method)
    # ========================================================================
    print(f"\nComputing target-source similarity (average method)...")
    
    model_target_avg_similarities = []
    
    for idx, row in model_results.iterrows():
        target = str(row.get('target', '')).strip()
        
        # Parse generated analogies
        try:
            generated_analogies_str = row.get('generated_analogies', '[]')
            if isinstance(generated_analogies_str, str):
                generated_analogies = json.loads(generated_analogies_str)
            else:
                generated_analogies = generated_analogies_str
            
            if not isinstance(generated_analogies, list):
                continue
                
            generated_analogies = [str(a).strip() for a in generated_analogies if str(a).strip() and str(a) != 'nan']
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            continue
        
        if not target or target == 'nan' or len(generated_analogies) == 0:
            continue
        
        # Compute similarity for each generated analogy
        target_similarities = []
        for analogy in generated_analogies:
            try:
                target_emb = embedder.encode([target.lower()])[0]
                analogy_emb = embedder.encode([analogy.lower()])[0]
                
                similarity = np.dot(target_emb, analogy_emb) / (
                    np.linalg.norm(target_emb) * np.linalg.norm(analogy_emb) + 1e-8
                )
                target_similarities.append(float(similarity))
            except Exception as e:
                continue
        
        if len(target_similarities) > 0:
            # Average similarities for this target
            avg_sim = np.mean(target_similarities)
            model_target_avg_similarities.append({
                'target': target,
                'avg_similarity': avg_sim,
                'num_analogies': len(target_similarities)
            })
    
    model_sim_df_recalculated = pd.DataFrame(model_target_avg_similarities)
    
    if len(model_sim_df_recalculated) > 0:
        print(f"  Computed average similarities for {len(model_sim_df_recalculated)} targets")
        print(f"  Mean: {model_sim_df_recalculated['avg_similarity'].mean():.4f}")
        print(f"  Median: {model_sim_df_recalculated['avg_similarity'].median():.4f}")
        
        # Create target-source comparison visualization
        fig, ax = plt.subplots(figsize=(12, 6))
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')
        
        # Plot both histograms
        ax.hist(scar_sim_df_recalculated['avg_similarity'], bins=50, alpha=0.7, edgecolor='black', 
                color='steelblue', label='Correct SCAR')
        ax.hist(model_sim_df_recalculated['avg_similarity'], bins=50, alpha=0.7, edgecolor='black', 
                color='coral', label=f'{model_name} ({mode_name})')
        
        # Add mean lines
        ax.axvline(scar_sim_df_recalculated['avg_similarity'].mean(), color='blue', linestyle='--', 
                  linewidth=2, label=f'SCAR Mean: {scar_sim_df_recalculated["avg_similarity"].mean():.3f}')
        ax.axvline(model_sim_df_recalculated['avg_similarity'].mean(), color='red', linestyle='--', 
                  linewidth=2, label=f'Model Mean: {model_sim_df_recalculated["avg_similarity"].mean():.3f}')
        
        # Add threshold line
        ax.axvline(SIMILARITY_THRESHOLD, color='orange', linestyle='-', linewidth=2, 
                  label=f'Threshold: {SIMILARITY_THRESHOLD}')
        
        # Add 33.33% coverage threshold for SCAR
        threshold_33_scar = scar_sim_df_recalculated['avg_similarity'].quantile(0.6667)
        ax.axvline(threshold_33_scar, color='purple', linestyle=':', linewidth=2, 
                  label=f'SCAR 1/3 coverage: {threshold_33_scar:.3f}')
        
        # Set axis spines
        for spine in ax.spines.values():
            spine.set_color('black')
            spine.set_linewidth(1)
        
        ax.set_xlabel('Target-Source Similarity Score', fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.set_title(f'Target-Source Similarity: {model_name} ({mode_name}) vs Correct SCAR', 
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        
        plt.tight_layout()
        
        # Save figure
        safe_model_name = sanitize_filename(model_name)
        filename = f'{safe_model_name}_{mode_name}_target_source_comparison.png'
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  Saved: {filename}")
        plt.close()
    else:
        print(f"  Warning: No target-source similarities computed for {model_name} - {mode_name}")
    
    # ========================================================================
    # SOURCE-TO-SOURCE SIMILARITY
    # ========================================================================
    print(f"\nComputing source-to-source similarity...")
    
    model_source_to_source_similarities = []
    
    # Group by target to find targets with multiple analogies
    for target in model_results['target'].unique():
        target_rows = model_results[model_results['target'] == target]
        if len(target_rows) == 0:
            continue
        
        # Get all generated analogies for this target (from all rows with same target)
        all_analogies = []
        for _, row in target_rows.iterrows():
            try:
                generated_analogies_str = row.get('generated_analogies', '[]')
                if isinstance(generated_analogies_str, str):
                    generated_analogies = json.loads(generated_analogies_str)
                else:
                    generated_analogies = generated_analogies_str
                
                if isinstance(generated_analogies, list):
                    all_analogies.extend([str(a).strip() for a in generated_analogies if str(a).strip() and str(a) != 'nan'])
            except (json.JSONDecodeError, KeyError, TypeError):
                continue
        
        # Remove duplicates while preserving order
        seen = set()
        unique_analogies = []
        for a in all_analogies:
            if a not in seen:
                seen.add(a)
                unique_analogies.append(a)
        
        if len(unique_analogies) < 2:
            continue
        
        # Compute pairwise similarity between all analogies for this target
        for i, analogy1 in enumerate(unique_analogies):
            for j, analogy2 in enumerate(unique_analogies):
                if i < j:  # Only compute each pair once
                    try:
                        analogy1_emb = embedder.encode([analogy1.lower()])[0]
                        analogy2_emb = embedder.encode([analogy2.lower()])[0]
                        
                        similarity = np.dot(analogy1_emb, analogy2_emb) / (
                            np.linalg.norm(analogy1_emb) * np.linalg.norm(analogy2_emb) + 1e-8
                        )
                        
                        model_source_to_source_similarities.append({
                            'target': target,
                            'analogy1': analogy1,
                            'analogy2': analogy2,
                            'similarity': float(similarity),
                            'num_analogies_for_target': len(unique_analogies)
                        })
                    except Exception:
                        continue
    
    model_source_sim_df = pd.DataFrame(model_source_to_source_similarities)
    
    if len(model_source_sim_df) > 0:
        print(f"  Computed {len(model_source_sim_df)} source-to-source similarity pairs")
        print(f"  Mean: {model_source_sim_df['similarity'].mean():.4f}")
        print(f"  Median: {model_source_sim_df['similarity'].median():.4f}")
        
        # Create source-to-source comparison visualization
        fig, ax = plt.subplots(figsize=(12, 6))
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')
        
        # Plot both histograms
        ax.hist(source_sim_df['similarity'], bins=50, alpha=0.7, edgecolor='black', 
                color='steelblue', label='Correct SCAR')
        ax.hist(model_source_sim_df['similarity'], bins=50, alpha=0.7, edgecolor='black', 
                color='coral', label=f'{model_name} ({mode_name})')
        
        # Add mean lines
        ax.axvline(source_sim_df['similarity'].mean(), color='blue', linestyle='--', 
                  linewidth=2, label=f'SCAR Mean: {source_sim_df["similarity"].mean():.3f}')
        ax.axvline(model_source_sim_df['similarity'].mean(), color='red', linestyle='--', 
                  linewidth=2, label=f'Model Mean: {model_source_sim_df["similarity"].mean():.3f}')
        
        # Add threshold line
        ax.axvline(SIMILARITY_THRESHOLD, color='orange', linestyle='-', linewidth=2, 
                  label=f'Threshold: {SIMILARITY_THRESHOLD}')
        
        # Add 33.33% coverage threshold for SCAR
        threshold_33_scar_ss = source_sim_df['similarity'].quantile(0.6667)
        ax.axvline(threshold_33_scar_ss, color='purple', linestyle=':', linewidth=2, 
                  label=f'SCAR 1/3 coverage: {threshold_33_scar_ss:.3f}')
        
        # Set axis spines
        for spine in ax.spines.values():
            spine.set_color('black')
            spine.set_linewidth(1)
        
        ax.set_xlabel('Source-to-Source Similarity Score', fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.set_title(f'Source-to-Source Similarity: {model_name} ({mode_name}) vs Correct SCAR', 
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        
        plt.tight_layout()
        
        # Save figure
        safe_model_name = sanitize_filename(model_name)
        filename = f'{safe_model_name}_{mode_name}_source_to_source_comparison.png'
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  Saved: {filename}")
        plt.close()
    else:
        print(f"  Warning: No source-to-source similarities computed for {model_name} - {mode_name}")

print(f"\n{'='*70}")
print("Processing Complete!")
print(f"{'='*70}")
print(f"Total combinations processed: {len(model_mode_combinations)}")
print(f"Output directory: {output_dir}")
print(f"Expected files: {len(model_mode_combinations) * 2} (2 per combination)")